# Recovering surfaces with Dleto

CC-BY Brooksbank, Kassabov, Wilson

This notebook demonstrates how to use Dleto to detect when tensor data clusters near a hidden embedded surface. The demonstration uses an embedded sphere for this purpose, but it will be clear how to replace it with a different surface.

## 0. Enable Python for Notebooks

First check that Python and Jupyter are installed. Install the Python package with its lab extra using the next step.


## 1. Installing Dleto

To load `opendleto` run the next cell. This assumes that you have cloned the full [Git Repository](https://github.com/thetensor-space/OpenDleto). The cell locates the checkout and installs any missing lab packages into the Python kernel selected for this notebook.


In [ ]:
# Colab opens the notebook without its repository files; fetch them once.
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [base / "OpenDleto" for base in candidates]
if os.environ.get("OPENDLETO_ROOT"):
    candidates.insert(0, Path(os.environ["OPENDLETO_ROOT"]).expanduser())
root = next((p for p in candidates if (p / "python/src/opendleto").is_dir()
             and (p / "labs/FPED_1720.csv").is_file()), None)
cloned = root is None
if cloned:
    root = Path.cwd() / "OpenDleto"
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main",
                           "https://github.com/thetensor-space/OpenDleto.git", str(root)])
if cloned or any(importlib.util.find_spec(name) is None
                 for name in ("numpy", "scipy", "plotly")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
                           str(root / "python") + "[lab]"])
    importlib.invalidate_caches()
sys.path.insert(0, str(root / "python/src"))

Now we can start using Dleto. If you have edited OpenDleto since opening this notebook, restart the Python kernel before running it: Python does not reload imported modules automatically.

In [ ]:
import numpy as np
from opendleto import (SymmetricOp, apply_transforms, nondeg,
                       plot_tensor, rand_surface_tensor,
                       randomize_tensor, side_by_side, stratify)

# NumPy generators keep each experiment reproducible. Julia and NumPy
# have different random streams, so individual coefficients will differ.

## 2. Embedding a sphere in a tensor

The following builds an array whose nonzero values congregate close to the surface of a sphere:  


In [ ]:
rng = np.random.default_rng(20261118)
a = b = c = 12.0
r = 10.0

# Choose f(x, y, z) to define the surface.
def f(x, y, z):
    return (x-a)**2 + (y-b)**2 + (z-c)**2

ds = (int(np.ceil(2*r+a)), int(np.ceil(2*r+b)), int(np.ceil(2*r+c)))

# Populate S near the surface defined by f. Julia coordinates are 1-based.
x, y, z = np.ogrid[1:ds[0]+1, 1:ds[1]+1, 1:ds[2]+1]
support = np.abs(f(x, y, z) - r**2) < 2
S = np.zeros(ds)
S[support] = rng.standard_normal(support.sum())
print(f"Grid: {S.shape}; surface points: {np.count_nonzero(S)}")
plot_tensor(S, title="Experiment Sphere", color="blue")

We now use randomization to scramble the tensor, thereby hiding its embedded surface structure, and then pass the resulting array to Dleto:

In [ ]:
S_rn, Xs = randomize_tensor(S, rng=np.random.default_rng(20261118))
assert np.allclose(apply_transforms(S, Xs), S_rn)

S_nd = nondeg(S_rn)
Ys = tuple(basis.T for basis in S_nd.bases)
assert np.allclose(apply_transforms(S_rn, Ys), S_nd.core)

S_recovery = stratify(S_nd.core, tol=1e-6, max_vectors=4)
S_strat, Zs = S_recovery.tensor, S_recovery.transforms
assert np.allclose(apply_transforms(S_nd.core, Zs), S_strat)
print(f"Nondegenerate dimensions: {S_nd.ranks}; verified derivations: {S_recovery.total_derivation_count} (including scalar gauges)")
print(f"Forbidden energy fraction: {S_recovery.forbidden_energy_fraction:.2e}")

In [ ]:
side_by_side(S, S_rn, S_strat,
             titles=("Original Surface", "Scrambled Surface", "Recovered Surface"),
             colors=("blue", "red", "green"))

We invite you to repeat the scrambled surface construction with alternative parameter values. You should keep in mind, however, that tensor complexity is a function of the *product* of its dimensions, so we recommend making modest parameter increases as you acclimate to the effect on runtime.


## 3. Recovering the embedded surface

If you experimented with different parameter settings you will have observed that certain choices produce outcomes that are more like the original surface than others. You may also be somewhat surprised by *all* of the plots Dleto returned in the previous cell. This calls for some commentary on underlying theory before we proceed further.

One consequence of the theory is that points lying on the same cross-section of the original sphere produce the same real eigenvalues in that axis. Dleto orders points according to these eigenvalues so that points on the same level set are permuted together. In the case of our sphere, the effect is that the recovered sphere resembles an octant of the original sphere. 

Further variability arises from how points are sampled on the original surface.  There is a way to sample points on a surface that essentially restricts the output of Dleto to just the cross-sectional permutation described above. One can assign 
values $(u_x,v_y,w_z)$ to each point $(x,y,z)$ on the surface in such a way that $u_x+v_y+w_z\approx 0$ precisely when $(x,y,z)$ is near the target surface.  

We illustrate the idea using a sphere with equation 
$$r^2=(x-a)^2+(y-b)^2+(z-c)^2$$
We can discretize the sphere by selecting the values `u[i]`, `v[j]` and `w[k]` 
according to the following rule:
$$\begin{aligned}
u[i] & \approx (i-a)^2-r^2/3\\
v[j] & \approx (j-b)^2-r^2/3\\
w[k] & \approx (k-c)^2-r^2/3
\end{aligned}
$$
Using these values we see that
$$\begin{aligned}
0 & \approx u[i]+v[j]+w[k]\\
  & \approx \left((i-a)^2-\frac{r^2}{3}\right)+\left((j-b)^2-\frac{r^2}{3}\right)+\left((k-c)^2-\frac{r^2}{3}\right).
\end{aligned}
$$
It follows that
$$0\approx u[i]+v[j]+w[k]\quad \Rightarrow \quad (i-a)^2+(j-b)^2+(k-c)^2\approx r^2.$$
Thus, the values `u[i]`, `v[j]`, `w[k]` encode a discretized approximation of a sphere of radius $r$ and center $(a,b,c)$.

We refer to this as **sampling from the densor space**, and the command `rand_surface_tensor` samples points in this manner:


This notebook uses a modest radius so the automatic derivation solve remains quick and repeatable; larger examples may need different solver settings.

In [ ]:
rng = np.random.default_rng(20261118)
a = b = c = 0.0
r = 10.0

indices = np.arange(0, int(r)+2)
Us = (indices-a)**2 - r**2/3
Vs = (indices-b)**2 - r**2/3
Ws = (indices-c)**2 - r**2/3

S_den = rand_surface_tensor(Us, Vs, Ws, 1.5, rng=rng).data
print(f"Grid: {S_den.shape}; surface points: {np.count_nonzero(S_den)}")
plot_tensor(S_den, title="Densor Sphere Points", color="blue")

Now we scramble our tensor sampled from the densor space and stratify it (after removing redundancy).

In [ ]:
S_den_rn, Xs = randomize_tensor(S_den, rng=np.random.default_rng(20261118))
assert np.allclose(apply_transforms(S_den, Xs), S_den_rn)

S_den_nd = nondeg(S_den_rn)
Ys = tuple(basis.T for basis in S_den_nd.bases)
assert np.allclose(apply_transforms(S_den_rn, Ys), S_den_nd.core)

S_den_recovery = stratify(S_den_nd.core, tol=1e-7, max_vectors=6)
S_den_strat, Zs = S_den_recovery.tensor, S_den_recovery.transforms
assert np.allclose(apply_transforms(S_den_nd.core, Zs), S_den_strat)
print(f"Nondegenerate dimensions: {S_den_nd.ranks}; verified derivations: {S_den_recovery.total_derivation_count} (including scalar gauges)")
print(f"Forbidden energy fraction: {S_den_recovery.forbidden_energy_fraction:.2e}")

Finally, we scramble our surface tensor and stratify.

In [ ]:
side_by_side(S_den, S_den_rn, S_den_strat,
             titles=("Original Surface", "Randomized Surface", "Recovered Surface"),
             colors=("blue", "red", "green"))

Compare the outcome of this experiment with that of the first experiment. You may observe that the surface recovered in the second experiment more closely resembles the original than did the recovered surface in the first experiment. Per the discussion preceding the second experiment, this discrepancy is a consequence of the different ways in which the two original sphere tensors were sampled.

# Modified chisels

Many situations call for a chisel with restrictions such as symmetric or hermitian matrices.  The sphere lab is a good place to witness this implication.


In [ ]:
a = b = c = 0.0
r = 10.0
indices = np.arange(0, int(r)+2)
Us = (indices-a)**2 - r**2/3
Vs = (indices-b)**2 - r**2/3
Ws = (indices-c)**2 - r**2/3

# Restrict the randomization to orthogonal transforms.
S_den_rn, Xs = randomize_tensor(S_den, rng=np.random.default_rng(20261119),
                                 kind="orthogonal")
assert np.allclose(apply_transforms(S_den, Xs), S_den_rn)
assert all(np.allclose(X.T @ X, np.eye(len(X))) for X in Xs)

side_by_side(S_den, S_den_rn,
             titles=("Original Surface", "Orthogonally Randomized Surface"),
             colors=("blue", "red"))

Now when we stratify we will restrict our attention to symmetric operators.

In [ ]:
S_den_sym_recovery = stratify(S_den_rn, operators=SymmetricOp(),
                               tol=1e-7, max_vectors=6)
S_den_sym, Xsym = S_den_sym_recovery.tensor, S_den_sym_recovery.transforms
assert np.allclose(apply_transforms(S_den_rn, Xsym), S_den_sym)
print(f"Verified symmetric derivations: {S_den_sym_recovery.total_derivation_count} (including scalar gauges)")
print(f"Forbidden energy fraction: {S_den_sym_recovery.forbidden_energy_fraction:.2e}")

side_by_side(S_den, S_den_rn, S_den_sym,
             titles=("Original Surface", "Randomized Surface",
                     "Stratified (Symmetric Operators)"),
             colors=("blue", "red", "green"))

## Julia/Python check

With the stated seeds and dimensions, both versions have 270 occupied points and a 21×21×21 nondegenerate tensor in the first experiment. Both have 42 occupied points and an 11×11×11 nondegenerate tensor in the densor experiment. The derivation counts, including the two scalar gauge directions, are 5, 5, and 8 for the first, densor, and symmetric experiments in both versions. NumPy and Julia use different random streams, so individual tensor coefficients and the plotted orientations need not coincide.
